# Pashto Text Dataset Cleaning for Model Fine-tuning

This notebook provides a comprehensive data cleaning process for Pashto text datasets in preparation for model fine-tuning. We'll go through multiple steps to ensure the data is properly cleaned, normalized, and ready for training.

## Overview of Steps:
1. Load all datasets
2. Inspect data structure
3. Handle missing values and duplicates
4. Clean and normalize text
5. Create a unified cleaned dataset
6. Export the cleaned dataset for fine-tuning

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import re
import glob
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # For progress bars
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Datasets

First, we'll find all the CSV files in the workspace and load them. Since there are multiple files with different ranges of data, we'll combine them into a single dataframe for efficient processing.

In [ ]:
# Find all CSV files in the workspace
csv_files = glob.glob('/workspaces/pashto-text-dataset/ZamAI_Pashto_Datasets/pashto_dataset_uncleaned_*.csv')
print(f"Found {len(csv_files)} CSV files.")

# Display the list of files
for i, file in enumerate(csv_files):
    print(f"{i+1}. {os.path.basename(file)}")

In [ ]:
# Function to load data from CSV files
def load_csv_data(file_path):
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Create an empty list to store all dataframes
all_dfs = []

# Load each CSV file into a dataframe and append to the list
for file in tqdm(csv_files):
    df = load_csv_data(file)
    if df is not None:
        # Add a column to indicate source file
        df['source_file'] = os.path.basename(file)
        all_dfs.append(df)

# Combine all dataframes into a single dataframe
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"Combined dataset shape: {combined_df.shape}")
else:
    print("No data loaded successfully.")
    combined_df = pd.DataFrame()

## 2. Inspect the Data

Now that we have loaded the data, let's examine its structure, check for missing values, duplicates, and other potential issues that need to be addressed.

In [ ]:
# Display information about the dataset
print("Basic information about the dataset:")
print(combined_df.info())

# Display the first few rows
print("\nSample rows from the dataset:")
combined_df.head()

In [ ]:
# Check for missing values
missing_values = combined_df.isnull().sum()
missing_percentage = (missing_values / len(combined_df)) * 100

# Create a DataFrame to display missing values statistics
missing_info = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage.round(2)
})

print("Missing values analysis:")
missing_info

# Visualize missing values
plt.figure(figsize=(10, 6))
plt.bar(missing_info.index, missing_info['Missing Percentage'])
plt.title('Percentage of Missing Values by Column')
plt.xlabel('Columns')
plt.ylabel('Missing Values (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Check for empty strings (which might not be captured as NaN)
empty_strings = {col: (combined_df[col].astype(str).str.strip() == '').sum() 
                for col in combined_df.columns}
empty_percentage = {col: (count / len(combined_df)) * 100 
                   for col, count in empty_strings.items()}

empty_info = pd.DataFrame({
    'Empty Strings': empty_strings,
    'Empty Percentage': [round(val, 2) for val in empty_percentage.values()]
}, index=empty_strings.keys())

print("Empty strings analysis:")
empty_info

# Check data types and column statistics
combined_df.describe(include='all')

## 3. Handle Missing Values and Empty Strings

Based on our analysis, let's handle missing values and empty strings in the dataset. We'll either remove rows with missing essential information or fill them with appropriate values.

In [ ]:
# Save the original row count for comparison
original_row_count = len(combined_df)
print(f"Original dataset has {original_row_count} rows")

# Function to check if a string is empty or contains only whitespace/newlines
def is_empty_or_whitespace(text):
    if pd.isna(text):
        return True
    if isinstance(text, str):
        return text.strip() == ''
    return False

# Drop rows where both title and content are empty or missing
cleaned_df = combined_df.copy()
empty_title_content = cleaned_df.apply(
    lambda row: is_empty_or_whitespace(row['title']) and is_empty_or_whitespace(row['content']), 
    axis=1
)
cleaned_df = cleaned_df[~empty_title_content]

# For rows with empty title but valid content, we'll keep them
empty_title_only = cleaned_df.apply(
    lambda row: is_empty_or_whitespace(row['title']) and not is_empty_or_whitespace(row['content']), 
    axis=1
)
# We could fill with a placeholder or extract from content, but for now we'll just note them
print(f"Rows with empty title but valid content: {empty_title_only.sum()}")

# For rows with empty content but valid title, we'll keep them as they might be useful for some tasks
empty_content_only = cleaned_df.apply(
    lambda row: not is_empty_or_whitespace(row['title']) and is_empty_or_whitespace(row['content']), 
    axis=1
)
print(f"Rows with empty content but valid title: {empty_content_only.sum()}")

# Report on the changes
print(f"Removed {original_row_count - len(cleaned_df)} rows with both empty title and content")
print(f"Remaining dataset has {len(cleaned_df)} rows")

## 4. Remove Duplicates

Let's identify and remove duplicate entries from the dataset. Duplicates can negatively impact model training by giving more weight to repeated examples.

In [ ]:
# Check for exact duplicates in content column
content_duplicates = cleaned_df.duplicated(subset=['content'], keep='first')
print(f"Found {content_duplicates.sum()} exact duplicate content entries")

# Check for exact duplicates in title column
title_duplicates = cleaned_df.duplicated(subset=['title'], keep='first')
print(f"Found {title_duplicates.sum()} exact duplicate title entries")

# Check for exact duplicates in both title and content
full_duplicates = cleaned_df.duplicated(subset=['title', 'content'], keep='first')
print(f"Found {full_duplicates.sum()} duplicate entries with same title and content")

# Remove duplicates based on content (more important than title for fine-tuning)
no_dupe_df = cleaned_df.drop_duplicates(subset=['content'], keep='first')
print(f"After removing content duplicates: {len(no_dupe_df)} rows remain")

## 5. Normalize Text Data

Now we'll clean and normalize the text data. This includes:
- Removing extra whitespace and newlines
- Handling special characters
- Creating Pashto-specific text normalization functions
- Creating a stopwords list for Pashto (since we couldn't find the stopwords.csv file)

In [ ]:
# Create a basic Pashto stopwords list
# This is a minimal list since we don't have the stopwords.csv file referenced in the original notebook
pashto_stopwords = [
    # Common Pashto stop words (pronouns, conjunctions, prepositions)
    'د', 'په', 'او', 'چې', 'له', 'نه', 'څخه', 'سره', 'ته', 'يې', 'هم', 'خو', 'کې', 'به',
    'دا', 'هغه', 'دې', 'کوم', 'څه', 'که', 'هر', 'يا', 'وي', 'شي', 'باندې', 'لاندې',
    'زه', 'ته', 'دی', 'دا', 'موږ', 'تاسو', 'هغوی', 'زما', 'ستا', 'زموږ', 'ستاسو', 'هغه'
]

pashto_stopwords_set = set(pashto_stopwords)

# Function to clean and normalize Pashto text
def normalize_pashto_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Replace multiple newlines with single space
    text = re.sub(r'\n+', ' ', text)
    
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove quotes (often found in the data)
    text = text.replace('"', '').replace('"', '')
    
    return text

# Advanced function for text preprocessing including stopword removal
def preprocessing(raw_text, remove_stopwords=True):
    if pd.isna(raw_text) or not isinstance(raw_text, str):
        return ""
    
    # Normalize the text first
    text = normalize_pashto_text(raw_text)
    
    # Keep only Pashto characters and spaces
    # The range ا-ی covers the Pashto/Arabic character set
    letters_only_text = re.sub(r'[^ا-یـ\s]', ' ', text)
    
    # If we don't want to remove stopwords, return here
    if not remove_stopwords:
        return letters_only_text.strip()
    
    # Split words and remove stopwords
    words = letters_only_text.split()
    meaningful_words = [w for w in words if w not in pashto_stopwords_set]
    
    # Join clean words into a string
    cleaned_text = " ".join(meaningful_words)
    
    return cleaned_text.strip()

# Test the normalization functions on a sample from our data
sample_text = no_dupe_df['content'].iloc[0] if not no_dupe_df.empty else "پشتو متن مثال"
print("Original text:")
print(sample_text)
print("\nNormalized text:")
print(normalize_pashto_text(sample_text))
print("\nPreprocessed text (with stopword removal):")
print(preprocessing(sample_text, remove_stopwords=True))

In [ ]:
# Apply text normalization to the dataset
print("Applying text normalization to the dataset...")

# Create new normalized columns
no_dupe_df['normalized_title'] = no_dupe_df['title'].apply(normalize_pashto_text)
no_dupe_df['normalized_content'] = no_dupe_df['content'].apply(normalize_pashto_text)

# Apply full preprocessing with stopword removal (optional - we'll keep both versions)
no_dupe_df['processed_title'] = no_dupe_df['title'].apply(lambda x: preprocessing(x, remove_stopwords=True))
no_dupe_df['processed_content'] = no_dupe_df['content'].apply(lambda x: preprocessing(x, remove_stopwords=True))

# Count empty results after normalization
empty_norm_title = (no_dupe_df['normalized_title'] == "").sum()
empty_norm_content = (no_dupe_df['normalized_content'] == "").sum()
empty_proc_title = (no_dupe_df['processed_title'] == "").sum()
empty_proc_content = (no_dupe_df['processed_content'] == "").sum()

print(f"Empty normalized titles: {empty_norm_title}")
print(f"Empty normalized content: {empty_norm_content}")
print(f"Empty processed titles (after stopword removal): {empty_proc_title}")
print(f"Empty processed content (after stopword removal): {empty_proc_content}")

# Display a few examples of the normalized text
print("\nExamples of normalized and processed text:")
sample_indices = min(3, len(no_dupe_df))
display_cols = ['normalized_title', 'normalized_content', 'processed_title', 'processed_content']
no_dupe_df[display_cols].head(sample_indices)

In [ ]:
# Filter out records with very short content (may not be useful for training)
# We'll define "useful" as having at least 10 characters after normalization
min_content_length = 10

# Check content lengths after normalization
no_dupe_df['norm_content_length'] = no_dupe_df['normalized_content'].apply(len)
no_dupe_df['norm_title_length'] = no_dupe_df['normalized_title'].apply(len)

# Display distribution of content lengths
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(no_dupe_df['norm_content_length'].clip(0, 1000), bins=50)
plt.title('Content Length Distribution (clipped at 1000 chars)')
plt.xlabel('Length')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(no_dupe_df['norm_title_length'].clip(0, 200), bins=50)
plt.title('Title Length Distribution (clipped at 200 chars)')
plt.xlabel('Length')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Filter to keep only records with sufficient content
sufficient_content_df = no_dupe_df[no_dupe_df['norm_content_length'] >= min_content_length]
print(f"Removed {len(no_dupe_df) - len(sufficient_content_df)} records with content length < {min_content_length} characters")
print(f"Remaining dataset size: {len(sufficient_content_df)} records")

## 6. Prepare Final Dataset for Model Fine-tuning

Now we'll prepare the final dataset for model fine-tuning. We'll:
- Select the appropriate columns
- Format the data as needed for fine-tuning
- Split into train/validation sets (optional)
- Save the cleaned dataset

In [ ]:
# Create final dataset with selected columns
# For fine-tuning, we'll use the normalized versions (not the processed ones with stopwords removed)
# as stopword removal might be too aggressive for some models

final_df = sufficient_content_df[['normalized_title', 'normalized_content', 
                                 'source_file']].copy()

# Rename columns to more generic names for model training
final_df = final_df.rename(columns={
    'normalized_title': 'title',
    'normalized_content': 'text'
})

# Create a basic prompt-completion format often used in fine-tuning
# You may adjust this format depending on your specific model requirements
final_df['prompt'] = final_df['title'].apply(lambda x: f"Title: {x}\n\nWrite an article based on this title:")
final_df['completion'] = final_df['text']

# Optional: Create a dataset format suitable for instruction tuning
final_df['instruction'] = "Write a Pashto language article based on the provided title."
final_df['input'] = final_df['title']
final_df['output'] = final_df['text']

# Display a sample of the final dataset
print("Sample of final dataset for fine-tuning:")
final_df[['prompt', 'completion']].head(2)

In [ ]:
# Split the data into training and validation sets (optional)
from sklearn.model_selection import train_test_split

# Use a 90/10 split for train/validation
train_df, val_df = train_test_split(final_df, test_size=0.1, random_state=42)

print(f"Training set size: {len(train_df)} examples")
print(f"Validation set size: {len(val_df)} examples")

## 7. Save Cleaned Dataset

Finally, we'll save the cleaned dataset in different formats suitable for model fine-tuning.

In [ ]:
# Create output directory if it doesn't exist
output_dir = '/workspaces/pashto-text-dataset/ZamAI_Pashto_Datasets/cleaned_data'
os.makedirs(output_dir, exist_ok=True)

# Save the complete cleaned dataset
final_df.to_csv(f"{output_dir}/pashto_cleaned_full_dataset.csv", index=False)
print(f"Saved complete dataset to {output_dir}/pashto_cleaned_full_dataset.csv")

# Save training and validation sets
train_df.to_csv(f"{output_dir}/pashto_cleaned_train.csv", index=False)
val_df.to_csv(f"{output_dir}/pashto_cleaned_val.csv", index=False)
print(f"Saved train/validation splits to {output_dir}/")

# Save in JSONL format (common for many fine-tuning APIs)
import json

# Function to save DataFrame as JSONL
def save_as_jsonl(df, filename, format_type='prompt_completion'):
    with open(filename, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            if format_type == 'prompt_completion':
                # Format for models like GPT-3.5, etc.
                data = {
                    "prompt": row['prompt'],
                    "completion": row['completion']
                }
            elif format_type == 'instruction':
                # Format for instruction-tuning models
                data = {
                    "instruction": row['instruction'],
                    "input": row['input'],
                    "output": row['output']
                }
            f.write(json.dumps(data, ensure_ascii=False) + '\n')

# Save in prompt-completion format
save_as_jsonl(train_df, f"{output_dir}/pashto_train_prompt_completion.jsonl", format_type='prompt_completion')
save_as_jsonl(val_df, f"{output_dir}/pashto_val_prompt_completion.jsonl", format_type='prompt_completion')

# Save in instruction format
save_as_jsonl(train_df, f"{output_dir}/pashto_train_instruction.jsonl", format_type='instruction')
save_as_jsonl(val_df, f"{output_dir}/pashto_val_instruction.jsonl", format_type='instruction')

print(f"Saved JSONL files for different fine-tuning approaches to {output_dir}/")

# Create a summary file
with open(f"{output_dir}/dataset_info.txt", 'w', encoding='utf-8') as f:
    f.write(f"Pashto Dataset Summary\n")
    f.write(f"Created on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"Original data size: {original_row_count} examples\n")
    f.write(f"Final data size: {len(final_df)} examples\n")
    f.write(f"Training set size: {len(train_df)} examples\n")
    f.write(f"Validation set size: {len(val_df)} examples\n\n")
    f.write(f"Cleaning steps applied:\n")
    f.write(f"- Removed rows with both empty title and content\n")
    f.write(f"- Removed duplicate content entries\n")
    f.write(f"- Normalized text data (whitespace, newlines, etc.)\n")
    f.write(f"- Filtered out very short content entries (<{min_content_length} chars)\n")
    
print(f"Saved dataset summary to {output_dir}/dataset_info.txt")

# Display a completion message
print("\n✅ Data cleaning and preparation complete!")
print(f"The cleaned data is ready for fine-tuning and can be found in the {output_dir}/ directory.")

## 8. Conclusion and Next Steps

We have successfully completed the data cleaning process for the Pashto text dataset. Here's a summary of what we accomplished:

1. Loaded multiple CSV files containing Pashto text data
2. Inspected the data structure and identified issues (missing values, duplicates)
3. Handled missing values and empty strings
4. Removed duplicate entries
5. Normalized and cleaned text data
6. Created a custom Pashto stopwords list for text preprocessing
7. Formatted the data for model fine-tuning
8. Split the data into training and validation sets
9. Saved the cleaned dataset in multiple formats

### Next Steps:

1. **Fine-tune a language model** using the cleaned dataset
2. **Evaluate the model performance** on the validation set
3. **Tune hyperparameters** to improve model quality
4. **Deploy the fine-tuned model** for your specific application

Note: The specific next steps will depend on your intended use case for the fine-tuned model.

In [ ]:
# Safely describe the DataFrame only if it has columns
if 'combined_df' in locals():
    if combined_df.empty or combined_df.columns.size == 0:
        print("DataFrame is empty or has no columns. Check your data loading and preprocessing steps.")
    else:
        display(combined_df.describe(include='all'))
else:
    print("combined_df is not defined. Please check your data loading step.")